# Canadian PII NER — end-to-end, local (generate data → train → evaluate)

Local rework of `canadian_pii_end2end.ipynb`: same pipeline — generate JSONL →
convert to spaCy `.spacy` → frozen warm-up → unfrozen fine-tune → evaluate →
smoke test — but against your own Python environment and this repo's checkout
instead of a Colab runtime. No Drive mount, no `files.upload()`/`files.download()`;
everything reads from and writes to the repo directory directly.

Optimizer blocks are real `[training.optimizer]` (LR actually applies), stage 2
sources both components from the frozen model, and step caps are safety
ceilings with patience doing the early stopping.

**Before running:** this trains an NER model — stage 2 alone is 8000 steps by
default. On CPU that can take a while; expect it to run long if you don't have
a GPU spaCy can use (`gpu_allocator` is set to `null` below, i.e. CPU).

## Setup — locate the repo root

In [8]:
import os
from pathlib import Path

cwd = Path.cwd()
if not (cwd / "data_generation").is_dir() and (cwd.parent / "data_generation").is_dir():
    os.chdir(cwd.parent)

cwd = Path.cwd()
assert (cwd / "data_generation" / "synthetic_ner_generator.py").is_file(), (
    f"Expected the repo root (with data_generation/, evaluation/, requirements.txt) "
    f"but landed in {cwd}. Open this notebook from the repo's notebooks/ folder, "
    f"or edit this cell to os.chdir() into the repo root manually."
)
print("Working dir:", cwd)


Working dir: /Users/vb/Desktop/PII Tool/Canadian-Personally-Identifiable-Information-PII-Recognizer-tool


## 1. Install dependencies

Installs into whatever Python environment this kernel is running in — activate
a venv/conda env first if you don't want this in your system Python.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_lg


UnboundLocalError: cannot access local variable 'child' where it is not associated with a value

--- Logging error ---
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/pexpect/pty_spawn.py", line 315, in _spawnpty
    return ptyprocess.PtyProcess.spawn(args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/IPython/utils/_process_posix.py", line 125, in system
    child = pexpect.spawn(self.sh, args=['-c', cmd])  # Vanilla Pexpect
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/pexpect/pty_spawn.py", line 205, in __init__
    self._spawn(command, args, preexec_fn, dimensions)
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/pexpect/pty_spawn.py", line 303, in _spawn
    self.ptyproc = self._spawnpty(self.args, env=self.env,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/ptyprocess/ptyprocess.py", line 269, 

OSError: [Errno 9] Bad file descriptor
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 807, in start
    self.io_loop.start()
  File "/opt/miniconda3/envs/pii/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/opt/miniconda3/envs/pii/lib/python3.11/asyncio/base_events.py", line 608, in run_forever
    self._run_once()
  File "/opt/miniconda3/envs/pii/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once
    handle._run()
  File "/opt/miniconda3/envs/pii/lib/python3.11/async

## 2. Confirm the data generator is present

In [ ]:
assert os.path.isfile("data_generation/synthetic_ner_generator.py"), \
    "data_generation/synthetic_ner_generator.py not found — check the repo checkout"
print("Generator found: data_generation/synthetic_ner_generator.py")


Generator found: data_generation/synthetic_ner_generator.py


## 3. Generate the synthetic corpus (JSONL + label.json)
Adjust sizes as you like; defaults come from the script. This overwrites `synth/`
if it already exists.

In [ ]:
!python data_generation/synthetic_ner_generator.py --output-dir synth --seed 42 \
    --train-size 12000 --validation-size 1600 --test-size 1600
!ls -la synth


Generating 15,200 synthetic NER examples (seed=42)...
  Train: 12,000  |  Validation: 1,600  |  Test: 1,600
  Negative ratio: 30%

Generating training split...
  Buckets — Canadian: 8,400  Negative: 3,600  Rehearsal: 0
Generating validation split...
  Buckets — Canadian: 1,120  Negative: 480  Rehearsal: 0
Generating test split...
  Buckets — Canadian: 1,120  Negative: 480  Rehearsal: 0

Generation complete in 0.69s (21,977.3 examples/sec)

Validating splits...
Writing output files...

Wrote 15,200 examples to synth/
All validations passed. Done.
total 4864
drwxr-xr-x@  6 vb  staff      192 Aug 13 23:29 .
drwxr-xr-x@ 18 vb  staff      576 Aug 13 23:29 ..
-rw-r--r--@  1 vb  staff     1206 Aug 13 23:29 label.json
-rw-r--r--@  1 vb  staff   260402 Aug 13 23:29 test.jsonl
-rw-r--r--@  1 vb  staff  1956033 Aug 13 23:29 train.jsonl
-rw-r--r--@  1 vb  staff   264670 Aug 13 23:29 validation.jsonl


## 4. Convert JSONL → spaCy `.spacy` (BIO tags → entity spans)

In [ ]:
import json, spacy
from spacy.tokens import Doc, DocBin, Span

with open("synth/label.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}
vocab = spacy.blank("en").vocab

def convert(src, dst):
    db = DocBin(); n = 0
    with open(src) as f:
        for line in f:
            ex = json.loads(line)
            words = ex["tokens"]
            tags = [id2label[int(t)] for t in ex["tags"]]
            doc = Doc(vocab, words=words)
            spans, i = [], 0
            while i < len(tags):
                t = tags[i]
                if t.startswith("B-"):
                    lab = t[2:]; j = i + 1
                    while j < len(tags) and tags[j] == f"I-{lab}": j += 1
                    spans.append(Span(doc, i, j, label=lab)); i = j
                else:
                    i += 1
            doc.ents = spans
            db.add(doc); n += 1
    db.to_disk(dst); print(f"{src} -> {dst}  ({n} docs)")

os.makedirs("corpus", exist_ok=True)
convert("synth/train.jsonl",      "corpus/train.spacy")
convert("synth/validation.jsonl", "corpus/valid.spacy")
convert("synth/test.jsonl",       "corpus/test.spacy")


synth/train.jsonl -> corpus/train.spacy  (12000 docs)
synth/validation.jsonl -> corpus/valid.spacy  (1600 docs)
synth/test.jsonl -> corpus/test.spacy  (1600 docs)


## 5. Stage-1 config (frozen) + train

In [ ]:
%%writefile base_config.cfg
[paths]
train = "corpus/train.spacy"
dev   = "corpus/valid.spacy"

[system]
gpu_allocator = null

[nlp]
lang = "en"
pipeline = ["tok2vec","ner"]

[components]

[components.tok2vec]
source = "en_core_web_lg"
component = "tok2vec"

[components.ner]
factory = "ner"

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 64
maxout_pieces = 2
use_upper = true
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy.Tok2VecListener.v1"
width = 96

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
max_epochs = 0
patience = 1600
max_steps = 3000
eval_frequency = 50
seed = 42
accumulate_gradient = 1
frozen_components = ["tok2vec"]
annotating_components = ["tok2vec"]

[training.optimizer]
@optimizers = "Adam.v1"
learn_rate = 0.001

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
size = 500
tolerance = 0.2
discard_oversize = false

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0

[initialize]
vectors = "en_core_web_lg"


Writing base_config.cfg


In [ ]:
!python -m spacy init fill-config base_config.cfg config_stage1.cfg
!python -m spacy debug config config_stage1.cfg
!python -m spacy train config_stage1.cfg --output ./output_frozen


^C

Aborted.
Usage: python -m spacy debug config [OPTIONS] CONFIG_PATH
Try 'python -m spacy debug config --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Invalid value for 'CONFIG_PATH': Path 'config_stage1.cfg' does not exist.    │
╰──────────────────────────────────────────────────────────────────────────────╯
Usage: python -m spacy train [OPTIONS] CONFIG_PATH
Try 'python -m spacy train --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Invalid value for 'CONFIG_PATH': Path 'config_stage1.cfg' does not exist.    │
╰──────────────────────────────────────────────────────────────────────────────╯


## 6. Stage-2 config (unfrozen, sources frozen model) + train

In [ ]:
%%writefile base_config_stage2.cfg
[paths]
train = "corpus/train.spacy"
dev   = "corpus/valid.spacy"

[system]
gpu_allocator = null

[nlp]
lang = "en"
pipeline = ["tok2vec","ner"]

[components]

[components.tok2vec]
source = "output_frozen/model-best"
component = "tok2vec"

[components.ner]
source = "output_frozen/model-best"
component = "ner"

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
max_epochs = 0
patience = 1600
max_steps = 8000
eval_frequency = 50
seed = 42
accumulate_gradient = 1
frozen_components = []
annotating_components = []

[training.optimizer]
@optimizers = "Adam.v1"
learn_rate = 0.0001

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
size = 500
tolerance = 0.2
discard_oversize = false

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0

[initialize]
vectors = "en_core_web_lg"


In [ ]:
!python -m spacy init fill-config base_config_stage2.cfg config_stage2.cfg
!python -m spacy debug config config_stage2.cfg
!python -m spacy train config_stage2.cfg --output ./output_stage2


## 7. Evaluate per-type on the held-out test set

In [ ]:
!python -m spacy evaluate ./output_stage2/model-best corpus/test.spacy --output metrics.json
import json; print(json.dumps(json.load(open("metrics.json"))["ents_per_type"], indent=2))


## 8. Smoke test
Runs `evaluation/smoke_test.py` against the freshly trained model (it defaults
to `./output_stage2/model-best`, which is exactly what stage 2 just wrote).

In [ ]:
assert os.path.isfile("evaluation/smoke_test.py"), "evaluation/smoke_test.py not found"
!python evaluation/smoke_test.py


## 9. Wrap up
The trained model is already on disk at `output_stage2/model-best` — nothing to
download. This just confirms it's there and optionally archives it as a zip for
copying elsewhere.

In [ ]:
import shutil

model_dir = Path("output_stage2/model-best")
assert model_dir.is_dir(), f"{model_dir} not found — did stage 2 training finish?"
size_mb = sum(f.stat().st_size for f in model_dir.rglob("*") if f.is_file()) / 1e6
print(f"Model ready at: {model_dir.resolve()}  ({size_mb:.1f} MB)")

shutil.make_archive("model_stage2", "zip", model_dir)
print("Also archived to model_stage2.zip")
